In [38]:
#Reaktor - Attribut: Temperatur (i celsius), 
# kontrollstavar_nivå (0% till 100%, där 100% är helt inskjutna och stoppar reaktionen), 
# anrikat_bränsle (mängd urvan kvar i procent)
#Matte/logik - Metoder: generera_värme(): 
# värmen ökar baserat på (100 - kontrollsvara_nivå) * bränsle
# om temperaturen blir för hög får man en hårdsmälta
#Kylsystem - Egenskaper: Vattenflöde (liter per sekund), pumpar_status(T/F)
#Matte/logik - kyl_ner(reaktor): sänker reaktorns temp baserat på vattenflöde.
# Ju mer vatten, desto mer värme transporteras till turbinen
#Turbin - Egenskaper: aktuell_effekt (Megawatt, MW)
# total_genererad_el (MWh)
#Matte/logik - producera_el(avledd_värme): räknar ut elektricitet
#Matten här kan vara en verklighetsgrad(t.ex. att 35% av den termiska värmen blir till el).

In [39]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [40]:
import random
import datetime
import requests
password="0ab7"
kontrollstavar_niva=100

In [41]:
def temp_check():
    while True:
        try: 
            temperatur=float(input("Vänligen ange en starttemperatur: "))
            if 0 < temperatur < 2000:
                return temperatur
            else:
                print("Temperaturen ej gilltig! Måste vara mellan 0 och 2000!")
        except ValueError:
            print("Vänligen skriv en siffra")
def bransle_check():
    while True:
        try:
            anrikat_bransle=float(input("Mängd uran kvar i procent: "))
            if 0 < anrikat_bransle <= 100:
                return anrikat_bransle
            else:
                print("Ej gilltigt värde! Måste vara mellan 0 och 100")
        except ValueError:
            print("Vänligen ange en siffra")
def flode_pump_check():
    while True:
        try:
            pumpar_status=int(input("Ja=1, Nej=2\nFungerar pumpen? "))
            if pumpar_status==1:
                pumpar_status=True
                while True:
                    try:
                        vattenflode=int(input("Vattenflöde (1-1500): "))
                        if 1 <= vattenflode <= 1500:
                            return vattenflode, pumpar_status
                        else:
                            print("Måste vara mellan 1 och 1500")
                    except ValueError:
                        print("Vänligen skriv en siffra")

            elif pumpar_status==2:
                pumpar_status=False
                vattenflode=0
                return vattenflode, pumpar_status
            else:
                print("Ej gilltigt värde!")
        except ValueError:
            print("Vänligen skriv en siffra")
def godkand_status_print(temperatur, anrikat_bransle, pumpar_status, vattenflode):
    print(f"Starttemperaturen: {temperatur:.1f}°C godkänd")
    print(f"Mängd uran kvar: {anrikat_bransle:.1f}% godkänd")
    print(f"Status: Pump igång: {pumpar_status}, aktuellt flöde: {vattenflode:.1f} l/s")
    

In [42]:
def hamta_dynamiskt_vinstmal():
    idag=datetime.date.today()
    ar=idag.strftime("%Y")
    manad_dag=idag.strftime("%m-%d")
    url=f"https://www.elprisetjustnu.se/api/v1/prices/{ar}/{manad_dag}_SE3.json"
    try:
        response=requests.get(url)
        data=response.json()
        priser=[period ["SEK_per_kWh"] * 100 for period in data]
        snittpris=sum(priser)/len(priser)
        print("Hämtade elpriser lyckades!")
        print(f"Dagens snittpris på elbörsen: {snittpris:.1f} öre/kWh")
        bas_mal=5000
        dynamiskt_mal=bas_mal + (snittpris * 25)
        return int(dynamiskt_mal), snittpris
    except Exception as e:
        print("Kunde inte hämta elpriser. Använder standardmål på 5000 MWh")
        return 5000, 75.0


In [43]:
mal, pris=hamta_dynamiskt_vinstmal()
print(mal, pris)

Hämtade elpriser lyckades!
Dagens snittpris på elbörsen: 131.2 öre/kWh
8279 131.18757291666665


In [44]:
while True:
    try:
        meny=int(input("Meny - inloggning: \n1: JENSEN\n2: Friläge\n3: Random\n4: Avsluta\n"))
        if meny==1:
            for i in range(5):
                user_password=input(f"Försök kvar: {5-i}\nVänligen ange lösenordet: ")
                if password==user_password:
                    temperatur=150
                    anrikat_bransle=5
                    pumpar_status=True
                    vattenflode=6
                    print("Lyckades logga in!")
                    godkand_status_print(temperatur, anrikat_bransle, pumpar_status, vattenflode)
                    break
        elif meny==2:
            temperatur=temp_check()
            anrikat_bransle=bransle_check()
            vattenflode, pumpar_status=flode_pump_check()
            godkand_status_print(temperatur, anrikat_bransle, pumpar_status, vattenflode)
        elif meny==3:
            temperatur=random.uniform(1, 1999)
            anrikat_bransle=random.uniform(1,100)
            pumpar_status=random.choice([True, False])
            if pumpar_status==True:
                vattenflode=random.uniform(1,1500)
            else:
                vattenflode=0
            godkand_status_print(temperatur, anrikat_bransle, pumpar_status, vattenflode)
        elif meny==4:
            print("Programmet avslutas...")
            break
        else:
            print("Ej gilltig inmatning")
    except ValueError:
        print("Du måste skriva en siffra!")

Starttemperaturen: 650.0°C godkänd
Mängd uran kvar: 54.0% godkänd
Status: Pump igång: False, aktuellt flöde: 0.0 l/s
Programmet avslutas...
